# Kaiser Window State for Quantum Phase Estimation
Adapted from Qualtran v0.7.0. Copyright 2024 Google LLC, Apache-2.0. Frozen commit `bcb17bb25360ed68ce3bc084645db3e929799389`.


In [ ]:
from collections import Counter
from collections.abc import Callable, Mapping
from dataclasses import replace
from math import pi
from typing import cast

import matplotlib.pyplot as plt
import numpy as np
import sympy
import torch
from IPython import display as notebook_display
from numpy.typing import NDArray

from flagquantum.algorithms.primitives.black_box_prepare import BlackBoxPrepare
from flagquantum.algorithms.primitives.kaiser_window import KaiserWindowState
from flagquantum.algorithms.primitives.luis_perina_state import LuisPerinaState
from flagquantum.algorithms.primitives.phase_estimation_window import (
    PhaseEstimationWindow,
    RectangularWindowState,
)
from flagquantum.algorithms.primitives.textbook_phase_estimation import (
    TextbookPhaseEstimation,
)
from flagquantum.circuit import Circuit
from flagquantum.core.ir import Instruction
from flagquantum.core.ir.program.signatures import Signature
from flagquantum.drawer.call_graph import export_call_graph_dot, render_call_graph_svg
from flagquantum.drawer.signature_graph import export_signature_graph_dot
from flagquantum.dynamic import DynamicCircuit
from flagquantum.experimental.dynamic import run_dynamic
from flagquantum.qec.resources.estimation import estimate_ftqc_resources

SVG = cast(Callable[[bytes], object], notebook_display.SVG)
display = cast(Callable[[object], None], notebook_display.display)


def show_signature(name: str, signature: Signature) -> None:
    if all(type(register.bitsize) is int for register in signature):
        dot = export_signature_graph_dot(signature, name)
    else:
        # Symbolic declaration labels retain width, encoding and output lifetime.
        rows = {}
        for register in signature:
            rows[register.name + " width"] = str(register.bitsize)
            rows[register.name + " side"] = str(register.side.name)
            rows[register.name + " encoding"] = str(register.dtype.encoding)
            rows.update({key: str(value) for key, value in register.dtype.parameters})
        dot = export_call_graph_dot(name, {}, annotations={name: rows})
    display(SVG(render_call_graph_svg(dot)))


def make_z_qpe(
    theta_value: float, window: PhaseEstimationWindow
) -> TextbookPhaseEstimation:
    def controlled_power(
        control: int,
        ports: Mapping[str, tuple[int, ...]],
        workspace: tuple[int, ...],
        classical_bits: tuple[int, ...],
        repetitions: int,
        adjoint: bool,
    ) -> tuple[Instruction, ...]:
        if workspace or classical_bits:
            raise ValueError("The direct controlled-phase emitter uses no work")
        angle = 2 * pi * theta_value * (-1 if adjoint else 1)
        return (
            Instruction("cphase", (control, ports["q"][0]), {"theta": angle}),
        ) * repetitions

    return TextbookPhaseEstimation(Signature.build(q=1), window, controlled_power)



## `KaiserWindowState`
Bloq to prepare a Kaiser window state for high confidence Quantum Phase Estimation.

Kaiser window states are optimal to minimize the probability of error outside a given
confidence interval.
Given the bitsize $m$ and parameter $\alpha$, the bloq prepares an $m$-bit state with
coefficients

$$
    \sum\limits_{x=-M}^{M}\frac{1}{2M} \frac{I_0\left(\pi\alpha\sqrt{1-(x/M)^2}\right)}{I_0\left(\pi\alpha\right)}\ket{x}
$$

where $M = 2^{m-1}$. See Ref[1] for more details.


#### Parameters
 - `bitsize`: Number of bits in the control register of QPE.
 - `alpha`: Shape parameter, determines trade-off between main-lobe width and side lobe level. 

#### References
 - [Analyzing Prospects for Quantum Advantage in Topological Data Analysis](https://arxiv.org/abs/2209.13581). Berry et al. (2022). Appendix D


In [ ]:
# KaiserWindowState retains exact tensor and separate quantized-child declarations.


### Example Instances

In [ ]:
kaiser_window_state_small = KaiserWindowState(5, 2)


In [ ]:
kaiser_window_state_symbolic = KaiserWindowState(*sympy.symbols("n, alpha"))


#### Graphical Signature

In [ ]:
show_signature("kaiser_window_state_small", kaiser_window_state_small.signature)
show_signature("kaiser_window_state_symbolic", kaiser_window_state_symbolic.signature)


### Call Graph

In [ ]:
kaiser_window_graph = kaiser_window_state_small.source_graph(
    root="KaiserWindowState", preparation=("RotationPreparation", 32, 5)
)
display(
    SVG(
        render_call_graph_svg(
            export_call_graph_dot(
                kaiser_window_graph.root, kaiser_window_graph.direct_calls
            )
        )
    )
)
display(kaiser_window_graph.leaf_counts)


## Example of QPE comparing different window functions
This comparison evaluates the original exact ideal window tensors with 100,000 samples per six-bit window. It explicitly initializes each window tensor, then appends the shared controlled-power and inverse-QFT body. It does not execute ideal Kaiser state preparation. The quantized physical child appears separately below. The Holevo expression retains the original notebook convention.


In [ ]:
# The source comparison contracts exact window tensors. The native comparison
# initializes those tensors explicitly, then executes the shared QPE body.
# This path does not physically prepare an ideal Kaiser window.


In [ ]:
def exact_window_coefficients(window: PhaseEstimationWindow) -> NDArray[np.float64]:
    width = window.m_bits
    if not isinstance(width, int) or width < 1:
        raise ValueError("Tensor comparison requires a positive concrete width")
    size = 2**width
    if isinstance(window, KaiserWindowState):
        return window.kaiser_state_coeff
    if isinstance(window, LuisPerinaState):
        return np.asarray(
            np.sqrt(2 / (size + 1))
            * np.sin(np.pi * (1 + np.arange(size)) / (size + 1)),
            dtype=np.float64,
        )
    if isinstance(window, RectangularWindowState):
        return np.asarray(np.ones(size) / np.sqrt(size), dtype=np.float64)
    raise ValueError("The notebook compares the three original window tensors")


def injected_tensor_prepare(
    ports: Mapping[str, tuple[int, ...]],
    workspace: tuple[int, ...],
    classical_bits: tuple[int, ...],
    adjoint: bool,
) -> tuple[Instruction, ...]:
    # The input state already contains the ideal tensor. This explicit identity
    # is a tensor-contraction boundary, not a state-preparation implementation.
    return ()


def construct_composite_boq(bloq: TextbookPhaseEstimation) -> Circuit:
    width = bloq.m_bits
    if not isinstance(width, int):
        raise ValueError("Resolve phase width before tensor comparison")
    coefficients = exact_window_coefficients(bloq.ctrl_state_prep)
    inputs = np.kron(coefficients, [0, 1])
    circuit = Circuit(
        width + 1,
        inputs=torch.tensor(inputs, dtype=torch.complex128),
        dtype=torch.complex128,
    )
    injected = BlackBoxPrepare((("qpe_reg", width),), (), injected_tensor_prepare)
    operation = replace(bloq, preparation=injected).make_on(
        phase=tuple(range(width)), target=(width,)
    )
    circuit.append(operation)
    return circuit


def simulate_theta_estimate(
    bloq: TextbookPhaseEstimation, n_samples: int
) -> NDArray[np.int64]:
    circuit = construct_composite_boq(bloq)
    # The final target is |1>. The upper bits are the original big-endian phase.
    samples = circuit.sample(
        n_samples, generator=torch.Generator().manual_seed(73), format="index"
    )
    return np.asarray((samples.reshape(-1) >> 1).numpy(), dtype=np.int64)


def holevo_variance(thetas: NDArray[np.float64]) -> float:
    # Preserve the source notebook's angle convention verbatim.
    mu = np.mean(np.cos(thetas - theta))
    return float(1 / mu**2 - 1)


In [ ]:
theta = 0.51234
n_samples = 100_000
m = 6
x_vals = [x / 2**m for x in range(2**m)]
qpe_textbook = make_z_qpe(theta, RectangularWindowState(m))
thetas_textbook = simulate_theta_estimate(qpe_textbook, n_samples)
qpe_sinstate = make_z_qpe(theta, LuisPerinaState(m))
thetas_sinstate = simulate_theta_estimate(qpe_sinstate, n_samples)
kaiser_window_state = KaiserWindowState.from_precision_and_delta(3, 1e-2)
alpha = kaiser_window_state.alpha
assert kaiser_window_state.bitsize == m
qpe_kaiser_state = make_z_qpe(theta, kaiser_window_state)
thetas_kaiser_state = simulate_theta_estimate(qpe_kaiser_state, n_samples)


In [ ]:
plt.rcParams["figure.figsize"] = (20, 10)

theta_counts_textbok = np.bincount(thetas_textbook) / n_samples
theta_counts_sinstate = np.bincount(thetas_sinstate) / n_samples
theta_counts_kaiser = np.bincount(thetas_kaiser_state) / n_samples
var_textbook = holevo_variance(thetas_textbook / 2**m)
var_sinstate = holevo_variance(thetas_sinstate / 2**m)
var_kaiser = holevo_variance(thetas_kaiser_state / 2**m)

plt.plot(
    np.array(x_vals[: len(theta_counts_textbok)]),
    theta_counts_textbok,
    label=r"TextbookQPE; var($\tilde{\phi}$)=" f"{var_textbook:.2e}",
)
plt.plot(
    np.array(x_vals[: len(theta_counts_sinstate)]),
    theta_counts_sinstate,
    label=r"SinStateQPE; var($\tilde{\phi}$)=" f"{var_sinstate:.2e}",
)
plt.plot(
    np.array(x_vals[: len(theta_counts_kaiser)]),
    theta_counts_kaiser,
    label=f"KaiserQPE({alpha=:0.3}); " r"var($\tilde{\phi}$)=" f"{var_kaiser:.2e}",
)
plt.vlines(
    theta, 0, 1, linestyles="--", label=f"True Phase $\phi$={theta}", colors="red"
)
plt.yscale("log", base=10)
plt.ylabel(f"Fraction of samples")
plt.xlabel(
    f"{m}-bit approximation of phase measured by running QPE. Should be close to {theta=}"
)
plt.title(
    f"QPE on Z phase(2 * {theta}) using {m=} bit control register and {n_samples} samples."
)
plt.legend()
plt.show()


## Separate quantized physical child

The source declares this finite-precision child independently from its ideal tensor, and supplies no physical Kaiser decomposition. The two-bit example below uses a negative phase-gradient input and executes the existing measured rotation child through ordinary append. Its workspace, primitive counts, declared model and ideal-tensor error are separate observations. Source-file tests retain every small measurement history and both directions.


In [ ]:
physical_window = KaiserWindowState(2, 2)
work, slots = physical_window.rotation_preparation().requirements(0)
negative_gradient = np.exp(-2j * np.pi * np.arange(4) / 4) / 2
zero_target = np.array([1, 0, 0, 0])
clean = np.zeros(2**work)
clean[0] = 1
initial = np.kron(np.kron(zero_target, negative_gradient), clean)
circuit = DynamicCircuit(4 + work, inputs=torch.tensor(initial), dtype=torch.complex128)
circuit.append(
    physical_window.make_on(
        qpe_reg=(0, 1),
        phase_gradient=(2, 3),
        workspace=tuple(range(4, 4 + work)),
        classical_bits=tuple(range(slots)),
    )
)
execution = run_dynamic(circuit, shots=1, seed=73, strategy="trajectory")
physical_joint = execution.final_states.detach().numpy().reshape(16, 2**work)
np.testing.assert_allclose(physical_joint[:, 1:], 0, atol=1e-12, rtol=0)
ideal_joint = np.kron(physical_window.kaiser_state_coeff, negative_gradient)
ideal_tensor_error = float(np.max(np.abs(physical_joint[:, 0] - ideal_joint)))
primitive_counts = dict(Counter(g.name for g in circuit.to_ir().instructions))
declared_model = estimate_ftqc_resources(physical_window.source_resource_model())
display(primitive_counts)
display(declared_model.logical)
display(ideal_tensor_error)
